# Solar Filament Segmentation Challenge 2026 — end-to-end pipeline

**Team:** TalhaBacak · **Final submission reproduced here:** `sub036_t020pox_bag5.csv`, the system the report describes.

This notebook is the *workflow*: every stage is one call into a documented module under [`src/`](../src). Nothing is implemented inline, per the organiser's modularity guidance (discussion 728302). Reading it top to bottom reproduces the submitted predictions.

## Three ways to run it

| mode | what runs | needs | hardware | wall clock |
|---|---|---|---|---|
| `FINAL_STAGE` (default) | bagged selector + keep gate on the shipped test candidate pool, the final submission rebuilt and compared byte for byte | this repository only | CPU | seconds |
| `REPRODUCE` | test-time inference of all 17 networks, fusion and featurisation, then the final stage | competition data + the released weights | 1 GPU (16 GB) | ~1.5 h |
| `RETRAIN` | everything from the raw competition data, including the 17 trained networks and the selector | competition data | 1 A100-80GB | ~40 h |

`FINAL_STAGE` is possible because the repository ships three small derived artefacts: the featurised test candidate pool (`outputs/eda/test_pool_t020pox.parquet`, no labels, 1.2 MB), the labelled out-of-fold training pool with its extra features, and the fitted selector bundle. The network weights (17 files, 0.9 GB) are a release asset, not part of the Git history.

## Pipeline in one picture

```
MAGFiLO GONG H-alpha (2048x2048 JPEG, 707 train / 180 test)
  |
  |-- DETECTOR LANE  YOLO11m-seg @1792, 5 folds, all annotator copies
  |        -> per-image instance candidates (conf >= 0.05)
  |
  |-- SEMANTIC LANE  U-Net(EfficientNet-B3) @1536, Dice+BCE+clDice
  |        12 members at test time: 5 clDice folds, 5 instance-balanced ('blob') folds,
  |        2 heavily augmented models
  |        -> probability field, components at p >= 0.2
  |
  +-> FUSION       cluster at IoU > 0.5, confidence-weighted vote 0.5, pixel-disjoint
        -> 42 features per candidate
        -> SELECTOR  two-stage HGB  p(match) x E[IoU|match], quantile-mapped,
                     mean of 5 image-bootstrap copies
        -> keep conf >= 0.40 and p_sel >= 0.60, one-pixel boundary ring removed
        -> submission CSV (pixel-disjoint RLE)
```


## 0. Configuration and environment

Every constant that changes a number is named here. Nothing below reads a hard-coded path.

In [ ]:
MODE = 'FINAL_STAGE'        # 'FINAL_STAGE', 'REPRODUCE' or 'RETRAIN'

# --- release archive: the 17 network weights (REPRODUCE mode). See README, 'Released weights'.
RELEASE_URL = 'https://github.com/talhabacak/solar-filament-segmentation-2026/releases/download/weights-v1/artifacts_v1.tar'

# --- production hyper-parameters (documented in the report; ledger docs/15)
DET_MODEL, DET_IMGSZ, DET_EPOCHS, DET_BATCH = 'yolo11m-seg.pt', 1792, 40, 2
SEM_ENCODER, SEM_IMGSZ, SEM_EPOCHS, SEM_BATCH, SEM_LR = 'timm-efficientnet-b3', 1536, 30, 2, 3e-4
CLDICE_W, CLDICE_WARMUP = 0.3, 5      # clDice weight and warm-up epochs
BLOB_RADIUS, BLOB_W = 40, 1.0         # instance-balanced member: local radius, weight
SEM_BLEND = 0.5                       # P = SEM_BLEND*blob + (1-SEM_BLEND)*clDice
FUSE_SEM_THR, FUSE_MIN_AREA, FUSE_VOTE = 0.2, 256, 0.5   # 0.2 replaced 0.3: +0.0071 out of fold
DET_CONF_FLOOR, R_MAX = 0.05, 0.97    # detector candidates entering fusion
SHRINK = 0.05                         # nominal; removes the one-pixel boundary ring (report section 3.5)
POLICY_CONF, POLICY_P = 0.40, 0.60    # keep gate: the metric's break-even, not a tuned value
SELECTOR = 'outputs/selector/selector_v5_shrunk_t020pox.joblib'   # single-fit 42-feature bundle (featurisation)
BAGGED = 'outputs/selector/selector_t020pox_bag5.joblib'          # the submitted selector: 5 image-bootstrap copies
TEST_POOL = 'outputs/eda/test_pool_t020pox.parquet'               # featurised test candidates, no labels
FINAL = 'outputs/submissions/sub036_t020pox_bag5.csv'               # the submitted file
TEST_RECIPE = 'recipe_blob5_50'       # test-time semantic ensemble (see cell 5)

import os, subprocess, sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT / 'src')); sys.path.insert(0, str(ROOT / 'src' / 'models'))
PY = chr(34) + sys.executable + chr(34)   # quoted: the interpreter path may contain a space
def run(cmd):
    print('$', cmd, flush=True)
    r = subprocess.run(cmd, shell=True, cwd=ROOT)
    if r.returncode: raise SystemExit(f'failed ({r.returncode}): {cmd}')
print(ROOT)

In [ ]:
# exact package versions used to produce the submission
run(f'{PY} -m pip install -r requirements.txt')
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())

In [ ]:
# REPRODUCE: fetch and unpack the released weights if they are not on disk
if MODE == 'REPRODUCE' and not pathlib.Path('outputs/runs/e013_yolo11m_f0_1792/weights/best.pt').exists():
    assert RELEASE_URL, 'set RELEASE_URL to the release asset (README, Released weights)'
    import tarfile, urllib.request
    urllib.request.urlretrieve(RELEASE_URL, 'artifacts_v1.tar')
    with tarfile.open('artifacts_v1.tar') as t:
        t.extractall('.', filter='data')


## 1. Data and the split

The competition data is 707 training images (2048x2048 grayscale JPEG, GONG H-alpha) with **8199 polygon annotations in 1154 annotator entries** — the same physical observation is annotated by up to three people — and 180 test images.

**Micro-decision: the split unit is the physical observation, not the annotation entry.** Two annotator copies of one image share every pixel, so an image-level split leaks. `data/splits/folds_v1.parquet` is a 5-fold `GroupKFold` over the physical file name, frozen on 2026-08-20 and never regenerated; every experiment in the ledger uses it.

In [ ]:
import pandas as pd
if MODE != 'FINAL_STAGE' and not pathlib.Path('data/MAGFiLO_1.0_Kaggle_2026').exists():
    run('kaggle competitions download -c filament-segmentation-2026 -p data -q')
    run('cd data && unzip -q -o filament-segmentation-2026.zip')
folds = pd.read_parquet('data/splits/folds_v1.parquet')
print(folds.groupby('fold').size().rename('images'))
print('unique physical files:', folds.file_name.nunique())


In [ ]:
# ground-truth cache: decode every COCO polygon once into RLE at 2048x2048
if MODE == 'RETRAIN':
    run(f'{PY} src/eda/build_gt_cache.py')
    run(f'{PY} src/eda/build_manifest.py')


## 2. Detector lane — YOLO11m-seg, 5 folds

**Why a detector at all.** The metric is instance-level Panoptic Quality, so the pipeline must emit *objects*. A semantic map alone merges neighbouring filaments into one component (measured: merges are 5.9% of all false negatives, fragments 2.4%).

**Micro-decisions**
- *Training targets are all annotator copies* (`export_yolo_allcopies.py`): each of the 1154 entries becomes its own sample, because the metric scores one prediction set against **every** copy, so the target distribution is the annotator distribution (measured +0.0051 PQ against medoid collapse).
- *Augmentation is physics-aware*: `mosaic=0, mixup=0, copy_paste=0` (they destroy the full-disc radial context), `degrees=10, translate=0.05, scale=0.1`, `fliplr=0.5` (single-class masks are chirality-agnostic), `flipud=0`, HSV off except `hsv_v=0.3` (grayscale input).
- *Resolution 1792*: swept 1024 -> 1536 -> 1792 -> 2048; 1792 is the last size that improves (2048 saturates, ledger docs/15).
- Batch 2, 40 epochs, `patience=15`, Ultralytics' `optimizer='auto'` (AdamW, lr0 0.01).

In [ ]:
if MODE == 'RETRAIN':
    run(f'{PY} src/data/export_yolo_allcopies.py')
    for k in range(5):
        run(f'{PY} src/models/train_yolo.py --fold {k} --model {DET_MODEL} --imgsz {DET_IMGSZ} --epochs {DET_EPOCHS} --batch {DET_BATCH} --name e013_yolo11m_f{k}_1792')

In [ ]:
# out-of-fold detections (train), pre-de-overlap at conf >= 0.05
if MODE == 'RETRAIN':
    for k in range(5):
        w = f'outputs/runs/e013_yolo11m_f{k}_1792/weights/best.pt'
        run(f'{PY} src/models/sweep_oof.py cache --weights {w} --fold {k} --imgsz {DET_IMGSZ} --tag _e013')
# test detections of all five fold models (the test-time detector ensemble)
if MODE in ('REPRODUCE', 'RETRAIN'):
    run(f'{PY} src/models/cross_cache.py --splits test --nms 0.5')


## 3. Semantic lane — two U-Net members per fold

**Member 1 (clDice).** U-Net with an ImageNet EfficientNet-B3 encoder and scSE decoder attention, input 1536 (a 2048 disc resized once; the cache stores it), loss `Dice + BCE + 0.3 * clDice` with clDice warmed up over 5 epochs. clDice (Shit et al., CVPR 2020) optimises the soft skeleton, i.e. connectivity, which is what a 1-2 px bridge failure costs on PQ but not on Dice.

**Member 2 (instance-balanced 'blob').** Same architecture, loss weighted per connected instance within a 40 px local radius, so a 300 px filament and a 3000 px one contribute equally. Small instances are where the reach is lost.

**Micro-decisions**
- AdamW lr 3e-4, weight decay 1e-4, OneCycle schedule, AMP, batch 2, 30 epochs.
- Augmentation: horizontal flip 0.5, rotation +-10 degrees about the disc centre (keeps r/R meaningful), gamma jitter 0.8-1.25. No crops: the disc must stay whole.
- Input channels: the grayscale frame replicated to 3 (`--channels rep`). Physics channels (local contrast, radial coordinate) were measured below baseline and are not used.
- Targets are again all annotator copies (`--target copies`).
- The out-of-fold probability map is written by `semantic_candidates.py` with flip TTA.

In [ ]:
if MODE == 'RETRAIN':
    run(f'{PY} src/data/build_semantic_cache.py --size {SEM_IMGSZ}')
    for k in range(5):
        run(f'{PY} src/models/train_semantic.py --fold {k} --imgsz {SEM_IMGSZ} --epochs {SEM_EPOCHS} --batch {SEM_BATCH} --lr {SEM_LR} --encoder {SEM_ENCODER} --cldice {CLDICE_W} --cldice-warmup {CLDICE_WARMUP} --cache --workers 4 --name sem_b3_f{k}_1536_cldice')
        run(f'{PY} src/models/semantic_candidates.py --weights outputs/runs_semantic/sem_b3_f{k}_1536_cldice/best.pt --fold {k} --tag _cldice')
    for k in range(5):
        run(f'{PY} src/models/train_blobloss_local.py --fold {k} --size {SEM_IMGSZ} --radius {BLOB_RADIUS} --blob-w {BLOB_W} --cldice {CLDICE_W} --name bloblocal_f{k}_s1536_w1.0_r40')
        run(f'{PY} src/models/semantic_candidates.py --weights outputs/runs_aug/bloblocal_f{k}_s1536_w1.0_r40/best.pt --fold {k} --tag _blob1536')

In [ ]:
# the production out-of-fold field: an equal blend of the two members
if MODE == 'RETRAIN':
    run(f'{PY} src/models/blend_semantic.py --folds 0,1,2,3,4 --a blob1536 --b cldice --out blob5ens --weight {SEM_BLEND}')

## 4. Fusion — from two lanes to one candidate pool

`build_fused_pool.py` turns each image's detector masks and semantic components into candidate instances and describes each with 32 features; ten photometric and orientation features are added by `extra_features.py` (42 in total).

**Micro-decisions**
- Detector candidates enter at `conf >= 0.05` and `r/R <= 0.97` (no ground-truth centroid lies beyond r 0.94; suppressing the outer ring is free).
- Semantic components are extracted at `p >= 0.2` with `min_area = 256` px (0.2 replaced 0.3: +0.0050 over five folds with the selector refitted on the new pool, CI [+0.0011, +0.0087]).
- Overlapping masks from the two lanes are clustered; a cluster's mask is the confidence-weighted vote at threshold 0.5 (`--vote 0.5`), which is between the union and the intersection.
- Cluster masks are made pixel-disjoint greedily in confidence order (the submission format forbids overlap).
- Features: geometry (log area, aspect, eccentricity, boundary complexity, skeleton length, width proxy), position (r/R, |latitude|), semantic support (mean/max/p90 probability, fraction over 0.5, a 25 px ring mean, contrast), detector support (per-model confidences, member IoU agreement, source counts) and image context (candidate density, semantic area fraction).

The pooled pool is the object every selector experiment in the ledger is measured on.

In [ ]:
# The labelled out-of-fold pool of this configuration is shipped (outputs/eda/fused_pool_all_blob5_t020_labelled.parquet
# with outputs/eda/extra_features_t020.parquet); rebuilding it needs the out-of-fold caches of a full retrain.
if MODE == 'RETRAIN':
    run(f'{PY} src/models/build_fused_pool.py --folds 0,1,2,3,4 --sem-tag _blob5ens --sem-thr {FUSE_SEM_THR} --vote {FUSE_VOTE} --det-tag _e013 --out fused_pool_all_blob5.parquet')


## 5. Selector — which candidates to keep

PQ charges 0.5 for a false positive and 0.5 for a miss, so the keep decision is the largest lever in the pipeline (+0.03 PQ over confidence thresholding).

**Micro-decisions**
- Masks are shrunk by 5% of their area *before* labelling and before the features that depend on area are recomputed — the deployed order, because shrinking moves which candidates clear IoU > 0.5 (measured +0.0128 alone).
- The score is two-stage over **42 features** (32 geometric/provenance plus ten photometric and orientation descriptors): `p(match) x E[IoU_sum | match]`, both HistGradientBoosting (classifier 300 iterations, lr 0.05, 31 leaves, min leaf 25, l2 1.0; regressor 300, lr 0.06, fitted on positives only).
- The product is mapped through the **training** score CDF, so the keep threshold is a quantile and means the same thing at deployment as in the sweep.
- **Image-bootstrap bagging.** The submitted selector is the mean of five (classifier, regressor) pairs, each fitted on a bootstrap sample of *images*; the same averaging is used inside the quantile map so the gate keeps its meaning (`fit_selector_bagged.py`). Locally +0.0010, CI [-0.0009, +0.0030]; first among our submissions in the leaderboard's unrounded ordering. Pooled cross-fitted PQ 0.4431.
- Keep gate: `conf >= 0.40` and `p_sel >= 0.60`. This is derived, not tuned: PQ makes a candidate worth keeping when its expected copy-weighted IoU sum exceeds `0.5 * copies * PQ`, which is 0.361 here, and the measured calibration crosses 0.361 exactly between the 0.55-0.60 band (0.364) and the 0.60-0.65 band (0.485). A looser gate was submitted and the leaderboard rejected it.

Honesty note kept in the report: out-of-fold AUC is 0.907 against 0.975 in-sample; every selector-side ablation in the ledger is measured with the fully out-of-fold protocol (`e115b_oof_threshold.py`), never with in-sample scores.

In [ ]:
if MODE == 'RETRAIN':
    run(f'{PY} src/models/fused_shrink_eval.py --pool fused_pool_all_blob5.parquet --shrink {SHRINK} --two-stage --tag blob5prodts')
# The submitted selector, refitted from the shipped labelled pool (CPU, about a minute). Verified 2026-11-15:
# the refit rebuilds the final submission byte for byte.
if MODE in ('REPRODUCE', 'RETRAIN'):
    run(f'{PY} src/models/fit_selector_bagged.py --out {BAGGED}')


## 6. Test-time inference and the submission

**Micro-decision: the test semantic field is an ensemble of model *types*, not seeds.** `recipe_blob5_50` = 0.5 x mean(5 blob folds) + 1/6 x mean(5 clDice folds) + 1/6 x heavy-aug@768 + 1/6 x heavy-aug@1536. Members are inferred at the size they were trained at (running a model at a larger input costs 0.007 pixel-Dice) with flip TTA.

The five fold models see the test set as an ensemble; no fold model has seen any test image, and the fold structure only ever constrained *validation*.

`submit_v2.py` runs the networks and the fusion with exactly the training-time constants and dumps the featurised test candidate pool. `submit_from_pool.py` then applies the bagged selector and the keep gate `conf >= 0.40, p_sel >= 0.60` to that pool and writes pixel-disjoint RLE.

In [ ]:
# GPU: networks -> probability field -> fusion -> featurised test pool (also writes the unbagged predecessor)
if MODE in ('REPRODUCE', 'RETRAIN'):
    run(f'{PY} src/models/submit_v2.py --out outputs/submissions/_unbagged_reproduced.csv --semantic {TEST_RECIPE} --conf {POLICY_CONF} --p-min {POLICY_P} --shrink {SHRINK} --sem-thr {FUSE_SEM_THR} --selector {SELECTOR} --dump-pool {TEST_POOL}')
# CPU: the final submission from the pool; --check compares it with the submitted file byte for byte
run(f'{PY} src/models/submit_from_pool.py --pool {TEST_POOL} --selector {BAGGED} --conf {POLICY_CONF} --p-min {POLICY_P} --out outputs/submissions/final_reproduced.csv --check {FINAL}')


In [ ]:
# format check: 0 overlapping pixels, valid RLE round-trip, sane instance areas
# (the validator lists the test images, so it runs where the competition data are present)
if pathlib.Path('data/MAGFiLO_1.0_Kaggle_2026/test/test_images').exists():
    run(f'{PY} src/models/validate_submission.py outputs/submissions/final_reproduced.csv')


In [ ]:
# agreement with the submitted file, as a set of run-length encodings
a, b = pd.read_csv(FINAL), pd.read_csv('outputs/submissions/final_reproduced.csv')
ra, rb = set(a.iloc[:, -1]), set(b.iloc[:, -1])
print(f'shipped {len(a)} rows, reproduced {len(b)} rows, identical RLE {len(ra & rb)} ({len(ra & rb) / len(a):.1%})')
# FINAL_STAGE: the file is byte-identical on any machine (verified 2026-11-15; scikit-learn 1.7.2).
# REPRODUCE, measured on the unbagged predecessor (report section 7): on the machine that produced it the rebuild
# from the checkpoints is byte-identical in all 1117 encodings; on a different GPU architecture 1091 of 1117
# encodings are identical, 25 more are the same mask with a pixel of coding difference, and exactly one instance
# is lost where a candidate falls across the keep gate through floating-point differences alone (+-0.0004 PQ).


## 7. Evaluation — the numbers the rubric asks for

The quantitative rubric is PQ plus the distributions of Dice, IoU and the one-to-many / many-to-one relations. These are computed out-of-fold with the exact competition scorer (`src/filament/pq.py`, verified against the organiser's self-evaluation notebook) and the fully out-of-fold selector, so they are honest estimates rather than in-sample ones.

In [ ]:
if MODE == 'RETRAIN':
    run(f'{PY} src/eda/e115b_oof_threshold.py')          # honest OOF pool + scores
    run(f'{PY} outputs/eda/_eval_error_taxonomy_honest.py')  # PQ/SQ/RQ + error taxonomy


In [ ]:
import json
# the honest out-of-fold decomposition of the shipped operating point
r = json.load(open('outputs/eda/pq_decomposition.json'))
b = r['bands']['EASY r 0.2-0.7']
print(f"easy disc (r 0.2-0.7): {b['n']} GT copies  SQ {b['SQ']:.4f}  RQ {b['RQ']:.4f}  PQ {b['PQ']:.4f}")
print(f"  loss from matching {b['loss_RQ']:.4f}   loss from shape {b['loss_SQ']:.4f}")
for name, x in r['bands'].items():
    print(f"  {name:18s} n {x['n']:5d}  SQ {x['SQ']:.3f}  RQ {x['RQ']:.3f}  PQ {x['PQ']:.3f}")
print(f"\nof the unmatched copies in that band, {r['near_miss']} ({r['near_miss_share']:.1%}) have a pool")
print("candidate above IoU 0.4 -- the misses are near-misses, not invisibility (report section 4.2)")
a = json.load(open('outputs/eda/error_atlas.json'))
print(f"\naddressable population: of {a['n_unambiguous']} copies every annotator drew and agreed on,")
print(f"we miss {a['n_unambiguous_missed']} ({a['n_unambiguous_missed']/a['n_unambiguous']:.1%}), worth about {a['headroom_pq']:+.4f} PQ")


## 8. What is *not* in the pipeline, and why

Every one of these was implemented and measured; the ledger ([`docs/15_ABLATION_LEDGER.md`](../docs/15_ABLATION_LEDGER.md)) carries the numbers and the pre-registered gates. They are listed because a reproducible method needs its negative space as much as its positive one.

| axis | best measurement | decision |
|---|---|---|
| native 2048 / tiled crops | -20 reached rows, -0.0011 in the pool | out |
| bigger encoders and decoders (B5, UPerNet, UNet++, wide decoder, GroupNorm, SiLU/GELU) | all within the seed floor; UNet++ is a better solo model (0.4041 vs 0.3999) and adds +0.0004 | out |
| transformer member (SegFormer @512 and @1536) | the only family measured to be decorrelated (error correlation 0.871 vs 0.926-0.940), +0.0026 on fold 0 and **-0.0052 on fold 1** | out |
| committee of arms as an ensemble member | fold-0 +0.0096 but fold-1 -0.0012 and three board losses | out |
| spine / width / boundary auxiliary heads | oracle-width mask 0.657 IoU vs 0.667 for the current masks | out |
| candidate-conditioned mask refiner, three formulations | all negative; the residual form that starts as the identity is the least bad at -0.015 | out |
| unmerge (watershed on detector or geometry seeds) | ceiling +0.0056, proposer 8 pay / 126 hurt | out |
| photometric boundary re-cut (Otsu, relative contrast) | -0.07 and -0.060 PQ | out |
| selector features: limb, band recalibration, image quality, orientation, context, threshold stability, magnetic PIL | -0.0017 .. +0.0010, all tied | out |
| selector loss: value-weighted, or direct E[IoU_sum] | -0.0009 and -0.0016 | out |
| co-temporal magnetogram / AIA 304 / neighbouring frame | real pixel-level information (+0.0146 AUC over a matched control, 0 when shuffled) and no PQ: teacher +0.0015/+0.0003, student -0.0000/-0.0006 | out |
| pretraining: GONG distillation, in-domain masked-image reconstruction | indistinguishable from a seed replicate on both folds | out |
| sampling: anti-aliased downsampling, bilinear upsampling, label threshold | -0.0000, +0.0006, label downsample measured lossless (IoU 0.915, zero deletions) | out |
| augmentation: elastic jitter, sphere-rotation centre-to-limb | +0.0003 and +0.0000 / -0.0036 | out |
| pool thresholds: clustering IoU, fusion vote, detector NMS, blend weight | production is a local optimum on every one, read without the selector | kept as is |
| semantic extraction threshold 0.2 with a refit 42-feature selector | +0.0050 over five folds, CI [+0.0011, +0.0087] | **shipped** |
| image-bootstrap bagging of the selector, five copies | +0.0010, CI [-0.0009, +0.0030]; first in the board ordering | **shipped (final submission)** |
| marginal-band cascade at quantile 0.85 on top of bagging | +0.0008, CI [-0.0011, +0.0026] | out |
| interior hole filling (annotation protocol conformity) | +0.0001 locally, below the champion in the board ordering | out |


## 9. Artefact manifest

Small artefacts are in the repository; the network weights are in the release archive. The hashes let the committee verify both. Training data and the competition CSV are not redistributed.


In [ ]:
import hashlib
IN_REPO = [BAGGED, SELECTOR, TEST_POOL, FINAL,
           'outputs/eda/fused_pool_all_blob5_t020_labelled.parquet', 'outputs/eda/extra_features_t020.parquet']
RELEASE = [f'outputs/runs/e013_yolo11m_f{k}_1792/weights/best.pt' for k in range(5)] + \
          [f'outputs/runs_semantic/sem_b3_f{k}_1536_cldice/best.pt' for k in range(5)] + \
          [f'outputs/runs_aug/bloblocal_f{k}_s1536_w1.0_r40/best.pt' for k in range(5)] + \
          ['outputs/runs_aug/aug_heavy_f0_768_60ep/best.pt', 'outputs/runs_aug/aug_heavy_f0_1536_60ep/best.pt']
for group, want in (('repository', IN_REPO), ('release archive', RELEASE)):
    print(group)
    for p in want:
        f = pathlib.Path(p)
        if f.exists():
            print(f'  {hashlib.sha256(f.read_bytes()).hexdigest()[:16]}  {f.stat().st_size/1e6:8.1f} MB  {p}')
        else:
            print(f'  {"MISSING":16}  {"":8}     {p}')
